# Task 1 - Problem & Dataset (Weeks 1-2)
**Module:** 7006SCN Machine Learning and Big Data
**Notebook:** `notebooks/Task1.ipynb`

**Aligned lectures:** Introduction to Big Data; Big Data Storage & Processing; Data Ingestion & Management.

**Dataset:** Amazon Reviews 2023 (McAuley Lab, UCSD) - **Cell_Phones_and_Accessories** category (~20M reviews). This is **unstructured text** data: the `text` field is free-form natural language requiring NLP before modelling.

**Problem:** binary **sentiment classification** from review text - positive (rating 4-5) vs negative (rating 1-2).

In [1]:
# === Colab bootstrap: install Spark (run once per session) ===
!pip -q install pyspark==3.5.1
import os, time
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .appName("7006SCN_AmazonReviews_Sentiment")
         .config("spark.sql.shuffle.partitions", "64")
         .config("spark.driver.memory", "12g")
         .config("spark.driver.maxResultSize", "2g")
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)
print("Spark UI:", spark.sparkContext.uiWebUrl)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 4.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 8.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.1 which is incompatible.
Spark version: 3.5.1
Spark UI: http://cac204c0d548:4040


In [2]:
# Mount Google Drive so data/models persist across the six notebooks.
from google.colab import drive
drive.mount('/content/drive')
BASE   = '/content/drive/MyDrive/7006SCN'
RAW    = BASE + '/raw'                      # downloaded jsonl.gz
PARQ   = BASE + '/reviews_parquet'          # raw -> parquet
PROC   = BASE + '/reviews_processed'        # after Task2 NLP pipeline
EXPORT = BASE + '/tableau'                  # Task6 exports
import os
for p in (BASE, RAW, EXPORT):
    os.makedirs(p, exist_ok=True)
print('Base:', BASE)

Mounted at /content/drive
Base: /content/drive/MyDrive/7006SCN


## 1.1 Download the raw data (CC BY-SA 4.0, public)
Spark reads gzipped JSON Lines directly. The single category file is multi-GB and holds ~20M reviews - well above the 10M requirement.

In [3]:
import urllib.request, os
CAT = "Cell_Phones_and_Accessories"
url = (f"https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/"
       f"raw/review_categories/{CAT}.jsonl.gz")
gz = f"{RAW}/{CAT}.jsonl.gz"
if not os.path.exists(gz):
    print("downloading (large, be patient) ...")
    req = urllib.request.Request(url, headers={"User-Agent":"Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=600) as r, open(gz,"wb") as f:
        f.write(r.read())
print("size GB:", round(os.path.getsize(gz)/1e9, 2))

downloading (large, be patient) ...
size GB: 2.53


## 1.2 SparkSession + load JSON, persist as Parquet

In [4]:
from pyspark.sql import functions as F
reviews = spark.read.json(gz)          # Spark parses jsonl.gz natively
print("Raw columns:", reviews.columns)

# Keep analysis-relevant columns (drop heavy 'images' list)
KEEP = ["rating","title","text","asin","parent_asin","user_id",
        "timestamp","helpful_vote","verified_purchase"]
df = reviews.select(*[c for c in KEEP if c in reviews.columns])

if not os.path.exists(PARQ):
    df.write.mode("overwrite").parquet(PARQ)
df = spark.read.parquet(PARQ)
print("Columns kept:", len(df.columns))

Raw columns: ['asin', 'helpful_vote', 'images', 'parent_asin', 'rating', 'text', 'timestamp', 'title', 'user_id', 'verified_purchase']
Columns kept: 9


## 1.3 Required evidence - `df.show(20)`, `printSchema()`, count, file size
**Screenshot the next four cells for the report.**

In [5]:
df.show(20, truncate=60)

+------+------------------------------------------------------------+------------------------------------------------------------+----------+-----------+----------------------------+-------------+------------+-----------------+
|rating|                                                       title|                                                        text|      asin|parent_asin|                     user_id|    timestamp|helpful_vote|verified_purchase|
+------+------------------------------------------------------------+------------------------------------------------------------+----------+-----------+----------------------------+-------------+------------+-----------------+
|   4.0|                            No white background! It’s clear!|I bought this bc I thought it had the nice white backgrou...|B08L6L3X1S| B08L6L3X1S|AFKZENTNBQ7A7V7UXW5JJI6UGRYQ|1612044451196|           0|             true|
|   5.0|                         Awesome!  Great price!  Works well!|Perfect. How pissed

In [6]:
df.printSchema()

root
 |-- rating: double (nullable = true)
 |-- title: string (nullable = true)
 |-- text: string (nullable = true)
 |-- asin: string (nullable = true)
 |-- parent_asin: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- helpful_vote: long (nullable = true)
 |-- verified_purchase: boolean (nullable = true)



In [7]:
n_rows = df.count(); n_cols = len(df.columns)
print(f"ROWS: {n_rows:,}   COLUMNS: {n_cols}")
assert n_rows >= 10_000_000, "Need >=10M rows."
print("Big Data row requirement: PASSED (>=10M)")

ROWS: 20,812,945   COLUMNS: 9
Big Data row requirement: PASSED (>=10M)


In [8]:
!du -sh "$PARQ" 2>/dev/null || true
size = sum(os.path.getsize(os.path.join(dp,f)) for dp,_,fs in os.walk(PARQ) for f in fs)
print(f"Parquet dataset size: {size/1e9:.2f} GB")

3.2G	/content/drive/MyDrive/7006SCN/reviews_parquet
Parquet dataset size: 3.40 GB


## 1.4 The 5 V's of Big Data
- **Volume** - ~20M reviews, multi-GB; text vectorisation explodes dimensionality, so distributed compute is essential.
- **Velocity** - Amazon reviews stream continuously; the batch pipeline is built to append new dumps.
- **Variety** - unstructured free text plus structured metadata (rating, helpful votes, verified-purchase flag, timestamp).
- **Veracity** - noisy text (typos, emojis, spam, sarcasm), missing titles, and rating/sentiment imbalance handled in Task 2/4.
- **Value** - automatic sentiment at scale powers product-quality monitoring, moderation, and recommendation - carried into the Tableau dashboards (Task 6).

## 1.5 Ethical & licensing note
Released by McAuley Lab under **CC BY-SA 4.0** (free use with attribution and share-alike). Reviews carry a pseudonymous `user_id` and free text that may contain personal information, so as a **data-minimisation / GDPR** measure I drop `user_id` and never attempt re-identification. Sentiment models can encode linguistic/demographic bias and the positive class dominates; bias and imbalance are examined in Tasks 4-5.

**Citation:** Hou, Y., Li, J., He, Z., Yan, A., Chen, X. and McAuley, J. (2024) *Bridging Language and Items for Retrieval and Recommendation*. arXiv:2403.03952. Dataset: https://amazon-reviews-2023.github.io/

## Version control
>=3 commits across Weeks 1-2, e.g. `Task1: SparkSession + jsonl.gz ingestion`, `Task1: parquet write + schema`, `Task1: 5Vs + licensing/ethics`.